# Using GEE to create a large set of analysis-ready data in tabular format

Before running this code, run script 01 to create the biotic stacks and then upload the results to GEE (as well as other disturbance stack stuff)

In [ ]:
import os

import ee
import geemap.foliumap as geemap
import geemap

from find_set_root import find_set_project_root
PROJECT_ROOT = find_set_project_root()
print(f"Project root found at: {PROJECT_ROOT}")
import utils.general_functions as ugf


DIR_RAW = os.path.join(PROJECT_ROOT, 'data', 'raw')
DIR_DERIVED = os.path.join(PROJECT_ROOT, 'data', 'derived')
ugf.dir_ensure([DIR_RAW, DIR_DERIVED])

#Prepare to use Earth Engine
ee.Authenticate()
ee.Initialize(project = 'ee-tymc5571-multi-disturbance')

#############
# USER SET PARAMETERS
#############

# Setup aoi
AOI_REGION = '6.2.14'
AOI_READABLE = "SRockies"
#AOI_READABLE = "test"

# Google drive folder
DRIVE_FOLDER = "GEE_Exports"

COARSE_FOREST_PERC_MIN = 0
COARSE_TRE_MAX_MIN = 25

FOREST_RANDOM_SAMPLE = 0.2
RANDOM_SEED = 1234
TILE_SIZE = 50000


Project root found at: C:\Users\tymc5571\dev\compound-disturbance-resilience
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\raw
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\derived


In [31]:
# Load datasets
treemap = ee.ImageCollection("USFS/GTAC/TreeMap/v2016")
chili = ee.Image("CSP/ERGo/1_0/US/CHILI")
lcms = ee.ImageCollection("USFS/GTAC/LCMS/v2023-9")
frg = ee.ImageCollection("LANDFIRE/Fire/FRG/v1_2_0")
tpi = ee.Image("CSP/ERGo/1_0/US/mTPI")
terraclimate = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")
srtm = ee.Image("USGS/SRTMGL1_003")
cbi = ee.Image("projects/ee-tymc5571-goodfire/assets/goodFire_all_cbi_bc_1985_2020")
lcmap = ee.ImageCollection("projects/sat-io/open-datasets/LCMAP/LCPRI")
rap = ee.ImageCollection("projects/rap-data-365417/assets/vegetation-cover-v3")
lf_bps = ee.ImageCollection("LANDFIRE/Vegetation/BPS/v1_4_0")
biotic = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/biotic_gridded_1km_all_years_severity")
vcf = ee.ImageCollection("MODIS/061/MOD44B")
hd_fingerprint = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_warm_fingerprint")
hd_z_def = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_def")
hd_z_pdsi = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_pdsi")
hd_z_vpd = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_vpd")
hd_z_tmmx = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_tmmx")
hd_z_soil = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_soil")  
hd_z_pr = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/hd_zscores_pr")



# AOI
epa_lvl3 = ee.FeatureCollection("users/tymc5571/EPA_EcoLvl3")
states = ee.FeatureCollection("TIGER/2018/States")

aoi = epa_lvl3.filter(ee.Filter.eq('NA_L3CODE', AOI_REGION))
# aoi= states.filter(ee.Filter.eq('STUSPS', 'CO'))
# aoi = ee.Geometry.Rectangle(
#     coords=[-105.545, 39.948, -105.445, 40.038],  # [xmin, ymin, xmax, ymax]
#     proj=None,
#     geodesic=False
# )

##########################

#Add CBI prefix if not present
cbi = cbi.rename(
    cbi.bandNames().map(
        lambda n: ee.Algorithms.If(
            ee.String(n).slice(0, 4).compareTo('cbi_').eq(0),
            n,
            ee.String('cbi_').cat(n)
        )
    )
)

# The temporal AOI will be based on the years in the CBI data selected
years = cbi.bandNames().map(lambda b: ee.Number.parse(ee.String(b).split('_').get(2)))
print(years.getInfo())
CBI_START_YEAR = ee.Number(years.sort().get(0)).getInfo()
CBI_END_YEAR = ee.Number(years.sort().get(-1)).getInfo()
print(CBI_START_YEAR, CBI_END_YEAR)


[1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
1985 2020


In [32]:
from typing import Literal, Any


# Utility functions

# -------------------------------
# Reclassify an image and mask all values except a specific value (e.g., forest class)
def reclassify_image_binary(value):
    def _reclassify(image):
        binary = image.eq(value).selfMask()
        return binary.copyProperties(image, ['system:time_start'])
    return _reclassify

# -------------------------------
# Combine two collections using logical AND (per image pair), copying time from the first image
def combine_collections_with_and(collection1, collection2, and_name):
    list1 = collection1.toList(collection1.size())
    list2 = collection2.toList(collection2.size())
    combined = list1.zip(list2)

    def combine_pair(pair):
        image1 = ee.Image(ee.List(pair).get(0))
        image2 = ee.Image(ee.List(pair).get(1))
        and_result = image1.And(image2).rename(and_name)
        return and_result.copyProperties(image1, ['system:time_start'])

    return ee.ImageCollection(combined.map(combine_pair))

# -------------------------------
# Add a 'year' property to each image
def add_year_property(image):
    year = image.date().get('year')
    return image.set('year', year)


def extract_single_band_value(image, collection, new_name):
    new_pts = image.reduceRegions(
        collection=collection,
        reducer=ee.Reducer.first(),
        scale=30
    ).map(lambda pt: pt.set(new_name, pt.get('first')).set('first', None))
    return new_pts

def reclassify_image_binary(value):
    def _reclassify(img):
        binary_img = img.eq(value).selfMask()
        return binary_img.copyProperties(img, ['system:time_start'])
    return _reclassify

def resample_to_coarse(source, coarse_image, reducer, max_pixels=4000):
    proj = coarse_image.projection()

    def reduce_and_reproject(img):
        img = img.setDefaultProjection(img.projection())
        return img.reduceResolution(
            reducer=reducer,
            maxPixels=max_pixels
        ).reproject(proj)

    return source.map(reduce_and_reproject) if isinstance(source, ee.ImageCollection) else reduce_and_reproject(source)


# def resample_to_coarse(source, coarse_image, reducer, max_pixels=4000):
#     # Derive valid projection from coarse_image
#     coarse_proj = ee.Image(coarse_image).projection()

#     def reduce_and_reproject(img):
#         img = ee.Image(img) \
#             .setDefaultProjection(coarse_proj)  # Ensure valid projection on input
#         reduced = img.reduceResolution(
#             reducer=reducer,
#             maxPixels=max_pixels
#         )
#         return reduced.reproject(coarse_proj)

#     if isinstance(source, ee.ImageCollection):
#         return source.map(reduce_and_reproject)
#     else:
#         return reduce_and_reproject(source)



def moving_window(
    image: ee.Image,
    window_size: int | float,
    reducer: ee.Reducer,
    kernel_shape: Literal[
        'square', 'circle', 'rect', 'cross', 'gaussian', 'manhattan', 'chebyshev'
    ] = 'square',
    keep_original_bandnames: bool = False,
    **kwargs: Any
) -> ee.Image:
    """
    Apply a moving window operation on an image using a specified reducer, window size, and kernel shape.

    Parameters:
    - image: ee.Image
    - window_size: int or float, diameter in meters
    - reducer: ee.Reducer (e.g., ee.Reducer.mean())
    - kernel_shape: One of ['square', 'circle', 'rect', 'cross', 'gaussian', 'manhattan', 'chebyshev']
    - keep_original_bandnames: If True, retain the original band names after filtering
    - kwargs: Additional parameters:
        - 'xRadius', 'yRadius' for 'rect'
        - 'sigma' for 'gaussian'
        - 'normalize' (bool) for all kernels (default: False)

    Returns:
    - ee.Image: Result of the focal/moving window operation
    """
    normalize: bool = kwargs.get('normalize', False)

    if kernel_shape == 'square':
        kernel = ee.Kernel.square(radius=window_size / 2, units='meters', normalize=normalize)
    elif kernel_shape == 'circle':
        kernel = ee.Kernel.circle(radius=window_size / 2, units='meters', normalize=normalize)
    elif kernel_shape == 'rect':
        xRadius = kwargs.get('xRadius', window_size / 2)
        yRadius = kwargs.get('yRadius', window_size / 2)
        kernel = ee.Kernel.rect(xRadius=xRadius, yRadius=yRadius, units='meters', normalize=normalize)
    elif kernel_shape == 'cross':
        kernel = ee.Kernel.cross(radius=window_size / 2, units='meters', normalize=normalize)
    elif kernel_shape == 'gaussian':
        sigma = kwargs.get('sigma', window_size / 6)
        kernel = ee.Kernel.gaussian(radius=window_size / 2, sigma=sigma, units='meters', normalize=normalize)
    elif kernel_shape == 'manhattan':
        kernel = ee.Kernel.manhattan(radius=window_size / 2, units='meters', normalize=normalize)
    elif kernel_shape == 'chebyshev':
        kernel = ee.Kernel.chebyshev(radius=window_size / 2, units='meters', normalize=normalize)
    else:
        raise ValueError(f"Unsupported kernel shape: {kernel_shape}")

    result = image.reduceNeighborhood(reducer=reducer, kernel=kernel)

    if keep_original_bandnames:
        original_names = image.bandNames()
        result = result.rename(original_names)

    return result


def remove_to_bands_append(image):
    """
    Renames bands in the input Earth Engine Image by removing the first part
    (before the first underscore) from each band name.

    Args:
        image (ee.Image): The input image.

    Returns:
        ee.Image: The image with renamed bands.
    """
    old_names = image.bandNames()
    new_names = old_names.map(lambda name: ee.String(name).split('_').slice(1).join('_'))
    return image.rename(new_names)

def prepend_or_append_to_band_names(
    image: ee.Image,
    text: str,
    mode: Literal["APPEND", "PREPEND"]
) -> ee.Image:
    """
    Appends or prepends a string to all band names of the given Earth Engine image.

    Args:
        image (ee.Image): The input Earth Engine image.
        text (str): The string to append or prepend.
        mode (Literal["APPEND", "PREPEND"]): Specifies whether to append or prepend the string.

    Returns:
        ee.Image: Image with updated band names.
    """
    old_names = image.bandNames()

    if mode == "APPEND":
        new_names = old_names.map(lambda name: ee.String(name).cat(text))
    elif mode == "PREPEND":
        new_names = old_names.map(lambda name: ee.String(text).cat(name))
    else:
        # This should never be reached due to Literal type, but it's good practice
        raise ValueError("Mode must be either 'APPEND' or 'PREPEND'")

    return image.rename(new_names)


def insert_to_band_names(image: ee.Image, text: str, index: int) -> ee.Image:
    """
    Inserts a string into each band name of an Earth Engine image at the specified index.
    
    Args:
        image (ee.Image): The input image.
        text (str): The string to insert.
        index (int): The character index at which to insert the string.
                     Supports negative indexing from the end.
    
    Returns:
        ee.Image: Image with updated band names.
    """
    def insert_at(name):
        name_str = ee.String(name)
        name_len = name_str.length()
        
        # Compute final insertion index (accounting for negative values)
        insert_pos = ee.Algorithms.If(
            index >= 0,
            ee.Number(index),
            name_len.add(index)  # index is negative, so subtract from length
        )
        insert_pos = ee.Number(insert_pos).clamp(0, name_len)  # avoid out-of-bounds
        
        prefix = name_str.slice(0, insert_pos)
        suffix = name_str.slice(insert_pos)
        return prefix.cat(text).cat(suffix)

    old_names = image.bandNames()
    new_names = old_names.map(insert_at)
    return image.rename(new_names)

def toBands_with_projection(collection):
    collection = ee.ImageCollection(collection)
    image = collection.toBands()
    reference_img = ee.Image(collection.first())
    return image.setDefaultProjection(reference_img.projection())

def filter_bands_by_year(image: ee.Image, first_year: int, last_year: int) -> ee.Image:
    band_names = image.bandNames()

    # Create list of allowed years as strings
    allowed_years = ee.List.sequence(first_year, last_year).map(
        lambda y: ee.Number(y).format('%04d')
    )

    # Map over band names and keep only those that match an allowed year suffix
    def keep_if_valid(band):
        band_str = ee.String(band)
        year_suffix = band_str.slice(-4)
        return ee.Algorithms.If(
            allowed_years.contains(year_suffix),
            band_str,
            None
        )

    # Map and filter out None
    valid_bands = band_names.map(keep_if_valid).removeAll([None])

    return image.select(valid_bands)

In [33]:
#################################
# PREP DATSETS
#################################

# Fire
fire_mask = cbi.mask().reduce(ee.Reducer.sum().unweighted()).gt(0).rename('valid')


# BIOTIC
# Load the image asset

#print("Available bands:", biotic.bandNames().getInfo())
new_biotic_names = [f'biotic_{year}' for year in range(1997, 2024)]
biotic = biotic.rename(new_biotic_names)
print("Renamed biotic bands:", biotic.bandNames().getInfo())



# biotic_sum = biotic.reduce(ee.Reducer.sum()).rename('biotic_sum')
# biotic_sum_1997_2007 = biotic.select(new_biotic_names[:11]).reduce(ee.Reducer.sum())

reference_proj = biotic.select([new_biotic_names[0]]).projection()
biotic_sum = biotic.reduce(ee.Reducer.sum()).setDefaultProjection(reference_proj)
biotic_sum_1997_2007 = biotic.select(new_biotic_names[:11]) \
    .reduce(ee.Reducer.sum()) \
    .setDefaultProjection(reference_proj)




###################
# Relaxed forest mask
###################

lcmap_cover = lcmap.filterDate(str(CBI_START_YEAR - 1), str(CBI_END_YEAR))
lcms_cover = lcms.filter(ee.Filter.eq('study_area', 'CONUS')).filterDate(str(CBI_START_YEAR - 1), str(CBI_END_YEAR)).select('Land_Cover')

lcmap_for = lcmap_cover.map(reclassify_image_binary(4))
lcms_for = lcms_cover.map(reclassify_image_binary(1))

combined_forest_mask = lcmap_for.map(lambda img: img.reduce(ee.Reducer.max()).gt(0)).max().Or(
    lcms_for.map(lambda img: img.reduce(ee.Reducer.max()).gt(0)).max()
)
combined_forest_mask = combined_forest_mask.setDefaultProjection(lcmap_cover.first().projection())

###################
# Conservative forest by year
###################

# Filter lcmap and lcms to relevant years and region
lcmap_cover_conus = lcmap.filterDate(str(CBI_START_YEAR - 1), str(CBI_END_YEAR))
lcms_cover_conus = (
    lcms
    .filter(ee.Filter.eq('study_area', 'CONUS'))
    .filterDate(str(CBI_START_YEAR - 1), str(CBI_END_YEAR))
    .select('Land_Cover')
)

# Reclassify to forest (value = 4 for lcmap, value = 1 for lcms)
lcmap_for = lcmap_cover_conus.map(reclassify_image_binary(4))
lcms_for = lcms_cover_conus.map(reclassify_image_binary(1))

# Combine both masks conservatively (AND)
conservative_forest_by_year = combine_collections_with_and(lcmap_for, lcms_for, 'conservativeForest').map(add_year_property)

###################
# Create distance from unburned forest
###################


print(conservative_forest_by_year.filter(ee.Filter.eq('year', years.get(0))).first().bandNames().getInfo())

# Get a list of all 'year' properties in conservative_forest
years_in_conservative_forest = conservative_forest_by_year.aggregate_array('year').getInfo()
print("Years in conservative_forest:", years_in_conservative_forest)

def unburned_forest_mask(year):
    band_name = ee.String('cbi_year_').cat(ee.String(year))
    cbi_year = cbi.select([band_name])
    forest_img = conservative_forest_by_year.filter(ee.Filter.eq('year', ee.Number(year).subtract(1))).first()
    return ee.Algorithms.If(
        forest_img,
        (lambda:
            (cbi_year.lt(1.25).unmask(0).Or(cbi_year.mask().Not()))
            .And(ee.Image(forest_img).mask())
            .selfMask()
            #.rename(ee.String('year_').cat(ee.String(year)).cat('_unburned_forest'))
            .rename(ee.String(year))
            .set('year', year)
        )(),
        None  # Skip year if forest_img is None; note that this will remove 1985, as the first year of forest cover data is 1985
    )

unburned_forest_images = years.map(unburned_forest_mask)
# Convert to ImageCollection
unburned_forest = ee.ImageCollection(unburned_forest_images)#.toBands()
unburned_forest = toBands_with_projection(unburned_forest)
unburned_forest = remove_to_bands_append(unburned_forest)
unburned_forest = insert_to_band_names(image=unburned_forest, text='unburned_forest_', index=0)


print(unburned_forest.bandNames().getInfo())

# GET DISTANCE
band_names = unburned_forest.bandNames()

# Define the function to apply to each band
def get_unburned_distance(band_name):
    band_name = ee.String(band_name)
    band = unburned_forest.select([band_name])
    transformed = band.fastDistanceTransform(30).sqrt().multiply(30)
    #return transformed.rename(band_name.cat('_distance'))
    return transformed.rename(band_name)

# Map over the band names and create an ImageCollection
transformed_bands = ee.ImageCollection(band_names.map(get_unburned_distance))

# Convert the ImageCollection to a single Image
unburned_forest_distance = toBands_with_projection(transformed_bands)#.toBands()
unburned_forest_distance = remove_to_bands_append(unburned_forest_distance)
unburned_forest_distance = insert_to_band_names(image=unburned_forest_distance, text='distance_', index=0)


print(unburned_forest_distance.bandNames().getInfo())


###################
# FRG
###################

# Reclassify FRG
frg_img = frg.filterMetadata('system:index', 'contains', 'CONUS').first()
frg_reclass = frg_img.remap(
    [1, 2, 3, 4, 5, 111, 112, 131, 132, 133],
    [1, 2, 3, 4, 5, 6, 6, 6, 6, 6],
    0, 'FRG'
).rename('frg_reclass')


###################
# HD
###################
hd_fingerprint = filter_bands_by_year(hd_fingerprint, CBI_START_YEAR, CBI_END_YEAR)

###################
# TREEMAP, BPS, and TERRACLIMATE
###################

# Process TreeMap and BPS
treemap = ee.ImageCollection(treemap)#.toBands().select(['TreeMap2016_Value', 'TreeMap2016_FLDTYPCD', 'TreeMap2016_FORTYPCD'])
treemap = toBands_with_projection(treemap).select(['TreeMap2016_Value', 'TreeMap2016_FLDTYPCD', 'TreeMap2016_FORTYPCD'])
lf_bps_img = lf_bps.filterMetadata('system:index', 'contains', 'CONUS').first()

# Terraclimate AET and DEF
terraclimate_mean = terraclimate.select(['aet', 'def']) \
    .filter(ee.Filter.calendarRange(1984, 2020, 'year')) \
    .mean()

# Define the functions to compute annual and summer terraclimate means
def annual_terraclimate_image(index):

    def annual_terraclimate_images(year):
        year = ee.Number(year)
        mean_image = terraclimate \
            .filter(ee.Filter.calendarRange(year, year, 'year')) \
            .select(index) \
            .mean() \
            .rename(ee.String(index).cat('_annual_').cat(year.format()))
        return mean_image

    annual_images = years.map(annual_terraclimate_images)
    annual_collection = ee.ImageCollection(annual_images)
    #annual_image = annual_collection.toBands()
    annual_image = toBands_with_projection(annual_collection)
    annual_image = remove_to_bands_append(annual_image)
    return(annual_image)

def summer_terraclimate_image(index):
    def summer_mean_image(year):
        year = ee.Number(year)
        # Filter for June (6), July (7), August (8) of that year
        summer_months = terraclimate \
            .filter(ee.Filter.calendarRange(year, year, 'year')) \
            .filter(ee.Filter.calendarRange(6, 8, 'month')) \
            .select(index) \
            .mean() \
            .rename(ee.String(index).cat('_summer_').cat(year.format()))
        return summer_months

    summer_images = years.map(summer_mean_image)
    summer_collection = ee.ImageCollection(summer_images)
    #summer_image = summer_collection.toBands()
    summer_image = toBands_with_projection(summer_collection)
    summer_image = remove_to_bands_append(summer_image)
    return summer_image


pdsi_annual = annual_terraclimate_image('pdsi')
pdsi_summer = summer_terraclimate_image('pdsi')
vpd_annual = annual_terraclimate_image('vpd')
vpd_summer = summer_terraclimate_image('vpd')

print("PDSI annual bands:", pdsi_annual.bandNames().getInfo())

###################
# RAP
###################
def rename_rap(img):
    year = ee.String(img.id().split('/').get(-1))
    new_names = img.bandNames().map(lambda b: ee.String('rap_').cat(b).cat('_').cat(year))
    return img.rename(new_names)

renamed_rap = rap.map(rename_rap)
#rap_mosaic = renamed_rap.toBands()
rap_mosaic = toBands_with_projection(renamed_rap)
fixed_names = rap_mosaic.bandNames().map(lambda b: ee.String(b).slice(5))
rap_mosaic = rap_mosaic.rename(fixed_names)


# Filter tree bands
tre_bands = rap_mosaic.bandNames().filter(ee.Filter.stringContains('item', 'TRE'))
rap_mosaic = rap_mosaic.select(tre_bands)



Renamed biotic bands: ['biotic_1997', 'biotic_1998', 'biotic_1999', 'biotic_2000', 'biotic_2001', 'biotic_2002', 'biotic_2003', 'biotic_2004', 'biotic_2005', 'biotic_2006', 'biotic_2007', 'biotic_2008', 'biotic_2009', 'biotic_2010', 'biotic_2011', 'biotic_2012', 'biotic_2013', 'biotic_2014', 'biotic_2015', 'biotic_2016', 'biotic_2017', 'biotic_2018', 'biotic_2019', 'biotic_2020', 'biotic_2021', 'biotic_2022', 'biotic_2023']
['conservativeForest']
Years in conservative_forest: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
['unburned_forest_1986', 'unburned_forest_1987', 'unburned_forest_1988', 'unburned_forest_1989', 'unburned_forest_1990', 'unburned_forest_1991', 'unburned_forest_1992', 'unburned_forest_1993', 'unburned_forest_1994', 'unburned_forest_1995', 'unburned_forest_1996', 'unburned_forest_1997', 'unburned_forest_19

In [34]:


########################
# CREATE MOVING WINDOW & RESAMPLED DATASETS
########################

# CBI Moving window
cbi_mw_median = moving_window(image=cbi, window_size=1000, kernel_shape='circle', reducer=ee.Reducer.median(), keep_original_bandnames=True)
cbi_mw_median = insert_to_band_names(image = cbi_mw_median, text='1kmmwmedian_', index=-9)

cbi_mw_std = moving_window(image=cbi, window_size=1000, kernel_shape='circle', reducer=ee.Reducer.stdDev(), keep_original_bandnames=True)
cbi_mw_std = insert_to_band_names(image=cbi_mw_std, text='1kmmwstd_', index=-9)


# RESAMPLE TO COARSE AND RENAME ALL
cbi_coarse = resample_to_coarse(cbi, biotic_sum, ee.Reducer.median())
cbi_coarse = insert_to_band_names(image=cbi_coarse, text='coarse_', index=-9)

cbi_mw_median_coarse = resample_to_coarse(cbi_mw_median, biotic_sum, ee.Reducer.median())
cbi_mw_median_coarse = insert_to_band_names(image=cbi_mw_median_coarse, text='coarse_', index=-9)

cbi_mw_std_coarse = resample_to_coarse(cbi_mw_std, biotic_sum, ee.Reducer.median())
cbi_mw_std_coarse = insert_to_band_names(image=cbi_mw_std_coarse, text='coarse_', index=-9)

unburned_forest_distance_coarse = resample_to_coarse(unburned_forest_distance, biotic_sum, ee.Reducer.median())
unburned_forest_distance_coarse = insert_to_band_names(image=unburned_forest_distance_coarse, text='coarse_', index=-4)

frg_reclass_coarse = resample_to_coarse(frg_reclass, biotic_sum, ee.Reducer.mode())
frg_reclass_coarse = insert_to_band_names(image=frg_reclass_coarse, text='coarse_', index=-4)

lf_bps_img_coarse = resample_to_coarse(lf_bps_img, biotic_sum, ee.Reducer.mode())
lf_bps_img_coarse = insert_to_band_names(image=lf_bps_img_coarse, text='coarse_', index=-4)

treemap_coarse = resample_to_coarse(treemap, biotic_sum, ee.Reducer.mode())
treemap_coarse = insert_to_band_names(image=treemap_coarse, text='coarse_', index=-4)

rap_mosaic_coarse = resample_to_coarse(rap_mosaic, biotic_sum, ee.Reducer.median())
rap_mosaic_coarse = insert_to_band_names(image=rap_mosaic_coarse, text='coarse_', index=-4)

srtm_coarse = resample_to_coarse(srtm, biotic_sum, ee.Reducer.median())
srtm_coarse = insert_to_band_names(image=srtm_coarse, text='coarse_', index=-4)

tpi_coarse = resample_to_coarse(tpi, biotic_sum, ee.Reducer.median())
tpi_coarse = insert_to_band_names(image=tpi_coarse, text='coarse_', index=-4)

chili_coarse = resample_to_coarse(chili, biotic_sum, ee.Reducer.median())
chili_coarse = insert_to_band_names(image=chili_coarse, text='coarse_', index=-4)




# ########################
# # CREATE COARSE PERCENTAGE COVER IMAGES - NOT WORKING!!! using forest_cover_max in the meantime
# ########################


                                # def get_perc(x, y):
                                #     # Ensure binary with values 0 and 1
                                #     x_bin = x.unmask(0).gt(0)

                                #     # Area of 1s
                                #     x_area = x_bin.multiply(ee.Image.pixelArea())

                                #     # Resample summed area to match Y
                                #     x_area_coarse = x_area \
                                #         .reduceResolution(reducer=ee.Reducer.sum(), maxPixels=4000) \
                                #         .reproject(y.projection())

                                #     # Total area of Y pixels
                                #     total_area = ee.Image.pixelArea().reproject(y.projection())

                                #     # Percentage cover
                                #     percent_covered = x_area_coarse.divide(total_area).multiply(100) \
                                #         .rename('percent_covered')

                                #     return percent_covered


                                # #fire_perc_coarse = resample_to_coarse(cbi.unmask(0).gt(0).multiply(ee.Image.pixelArea()), biotic_sum, ee.Reducer.sum())
                                # #fire_perc_coarse = get_perc(cbi.unmask(0).gt(0), biotic_sum)
                                # #fire_perc_coarse = insert_to_band_names(image=fire_perc_coarse, text='coarse_pixcount_', index=-9)

                                # # forest_perc_coarse = (resample_to_coarse(combined_forest_mask, biotic_sum, ee.Reducer.sum())
                                # #                     #   .multiply(900)
                                # #                     #   .divide(1000000)
                                # #                     #   .multiply(100)
                                # #                       .rename('forest_perc_coarse')
                                # # )

                                # #forest_perc_coarse = get_perc(combined_forest_mask.unmask(0).gt(0), biotic_sum).rename('forest_perc_coarse')

forest_cover_max = rap_mosaic_coarse.reduce(ee.Reducer.max()).rename('forest_cover_max')








# # Testing
# print(cbi_mw_median_coarse.bandNames().getInfo())

# Map = geemap.Map(center=[0, 0], zoom=2)
# #Map.addLayer(fire_perc_coarse.select('cbi_coarse_pixcount_year_2020'), {'min': 0, 'max': 100, 'palette': ['blue', 'red']}, 'fire_perc')
# Map.addLayer(cbi.select('cbi_year_2020'), {'min': 0, 'max': 3, 'palette': ['blue', 'green', 'yellow', 'orange', 'red']}, 'CBI')
# Map.addLayer(cbi_coarse.select('cbi_coarse_year_2020'), {'min': 0, 'max': 3, 'palette': ['blue', 'green', 'yellow', 'orange', 'red']}, 'CBI Coarse')
# # Map.addLayer(unburned_forest.select('unburned_forest_2020'), {'min': 0, 'max': 1, 'palette': ['red', 'green']}, 'Unburned Forest')
# # Map.addLayer(unburned_forest_distance.select('distance_unburned_forest_2020'), {'min': 0, 'max': 500, 'palette': ['red', 'yellow', 'green']}, 'Unburned Distance')
# Map.addLayer(cbi_mw_median.select('cbi_1kmmwmedian_year_2020'), {'min': 0, 'max': 3, 'palette': ['blue', 'green', 'yellow', 'orange', 'red']}, 'Test CBI median window')
# Map.addLayer(cbi_mw_median_coarse.select('cbi_1kmmwmedian_coarse_year_2020'), {'min': 0, 'max': 3, 'palette': ['blue', 'green', 'yellow', 'orange', 'red']}, 'Test CBI median window coarse')
# Map.addLayer(combined_forest_mask, {'min': 0, 'max': 1, 'palette': ['red', 'green']}, 'Conservative Forest 2019')
# Map.addLayer(forest_perc_coarse, {'min': 0, 'max': 100, 'palette': ['white', 'green']}, 'Forest % (Coarse)')
# # Map.addLayer(biotic_sum_1997_2007, {'min':0, 'max':100, 'palette': ['blue', 'yellow','red']}, 'Sum 1997-2007')
# Map.addLayer(forest_cover_max, {'min': 0, 'max': 100, 'palette': ['red', 'green']}, 'RAP Mosaic Coarse')
# Map





In [35]:
# CREATE SAMPLE

# Mask to the ROI
coords_img = ee.Image.pixelCoordinates(biotic_sum.projection())
masked_coords = coords_img.updateMask(biotic_sum.mask())

# Sample pixel centers (creates FeatureCollection with geometries)
sample_points = masked_coords.sample(
    region=aoi,
    scale=1000,
    geometries=True
)


# print('test sample size:', sample_points.size().getInfo())

# Map = geemap.Map(center=[0, 0], zoom=2)
# Map.addLayer(biotic_sum_1997_2007, {'min': 0, 'max': 100, 'palette': ['blue', 'yellow', 'red']}, 'Biotic Sum 1997-2007')
# Map.addLayer(sample_points, {}, 'Sample Points')
# Map

In [36]:
# Map = geemap.Map(center=[40.0, -105.5], zoom=11)

# # Add forest percentage image
# Map.addLayer(forest_perc_coarse, {
#     'min': 0, 'max': 100, 'palette': ['white', 'green']
# }, 'Forest % (Coarse)')

# Map.addLayer(fire_mask, {
#     'min': 0, 'max': 1, 'palette': ['white', 'red']
# }, 'Fire Mask')

# # Add forest mask
# Map.addLayer(combined_forest_mask.updateMask(combined_forest_mask), {
#     'palette': ['darkgreen']
# }, 'Combined Forest Mask')

# # Add coarse sample points
# #Map.addLayer(fc_coarse, {}, 'Coarse Sample Points')

#Map


In [37]:
# EXTRACTION

fc = sample_points
fc_coarse = sample_points

# Do direct extraction and filter of forest and fire
# fc = extract_single_band_value(combined_forest_mask, fc, 'forest').filter(ee.Filter.eq('forest', 1))
# fc = extract_single_band_value(fire_mask, fc, 'fire')

fire_only = fc.filter(ee.Filter.And(
    ee.Filter.eq('forest', 1),
    ee.Filter.eq('fire', 1)
))
forest_only = fc.filter(ee.Filter.And(
    ee.Filter.eq('forest', 1),
    ee.Filter.eq('fire', 0)
))

forest_sample = forest_only.randomColumn('rand', seed=RANDOM_SEED).filter(ee.Filter.lt('rand', FOREST_RANDOM_SAMPLE))
combined = fire_only.merge(forest_sample)

# Add variables to points
# Define (image, label) pairs
image_label_pairs = [
    (srtm, 'srtm'),
    (tpi, 'tpi'),
    (frg_reclass, 'frg_reclass'),
    (chili, 'chili'),
    (lf_bps_img, 'lf_bps'),
    (biotic_sum, 'biotic_sum')
]

# Apply the extraction function
for img, label in image_label_pairs:
    fc = extract_single_band_value(img, fc, label)



# Add multi-band data using reduceRegions
# List of images to reduce
multiband_images = [cbi, biotic, rap_mosaic, treemap, terraclimate_mean, \
                    pdsi_annual, pdsi_summer, \
                    # vpd_annual, vpd_summer, \
                    hd_fingerprint, \
                    # hd_z_def, hd_z_pdsi, hd_z_vpd, hd_z_tmmx, hd_z_soil, hd_z_pr, \
                    cbi_mw_median, cbi_mw_std, unburned_forest_distance]

# Apply reduceRegions for each image
for img in multiband_images:
    fc = img.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=30)


# Do coarse extraction
#fc_coarse = extract_single_band_value(forest_perc_coarse, fc_coarse, 'forest').filter(ee.Filter.gt('forest', COARSE_FOREST_PERC_MIN))
#fc_coarse = extract_single_band_value(forest_cover_max, fc_coarse, 'forest').filter(ee.Filter.gt('forest', COARSE_TRE_MAX_MIN))


# Add variables to coarse points
# Define (image, label) pairs for coarse extraction
image_label_pairs_coarse = [
    #(fire_perc_coarse, 'fire'),
    (srtm_coarse, 'srtm'),
    (tpi_coarse, 'tpi'),
    (frg_reclass_coarse, 'frg_reclass'),
    (chili_coarse, 'chili'),
    (lf_bps_img_coarse, 'lf_bps'),
    (biotic_sum, 'biotic_sum')
]


# Apply the extraction function
for img, label in image_label_pairs_coarse:
    fc_coarse = extract_single_band_value(img, fc_coarse, label)


# Multiband extraction for coarse data
multiband_images_coarse = [ cbi_coarse, biotic, rap_mosaic_coarse, treemap_coarse, terraclimate_mean, \
    pdsi_annual, pdsi_summer, \
    # vpd_annual, vpd_summer, \
    hd_fingerprint, \
    # hd_z_def, hd_z_pdsi, hd_z_vpd, hd_z_tmmx, hd_z_soil, hd_z_pr, \
    cbi_mw_median_coarse, cbi_mw_std_coarse, unburned_forest_distance_coarse]

# Apply reduceRegions for each image
for img in multiband_images_coarse:
    fc_coarse = img.reduceRegions(collection=fc_coarse, reducer=ee.Reducer.first(), scale=30)





# Round to minimize string sizes in CSVs
def round_fc(feature_collection):
    def round_feature(feat):
        props = feat.toDictionary()
        keys = ee.List(props.keys())

        def keep_if_tenths(key):
            key_str = ee.String(key)
            cond1 = key_str.slice(0, 7).equals('biotic_')
            cond2 = key_str.slice(0, 4).equals('cbi_')
            return ee.Algorithms.If(cond1, key, ee.Algorithms.If(cond2, key, None))

        def keep_if_ints(key):
            key_str = ee.String(key)
            cond1 = key_str.slice(0, 4).equals('aet')
            cond2 = key_str.slice(0, 4).equals('def')
            cond3 = key_str.slice(0, 9).equals('distance_')
            cond4 = key_str.slice(0, 5).equals('pdsi_')
            cond5 = key_str.slice(0, 4).equals('vpd_')
            return ee.Algorithms.If(cond1, key,
                ee.Algorithms.If(cond2, key,
                ee.Algorithms.If(cond3, key,
                ee.Algorithms.If(cond4, key,
                ee.Algorithms.If(cond5, key, None)))))

        round_tenths = keys.map(keep_if_tenths).removeAll([None])
        round_int = keys.map(keep_if_ints).removeAll([None])

        def round_tenths_fn(key):
            val = ee.Number(props.get(key)).multiply(10).round().divide(10)
            return ee.List([key, val])

        def round_int_fn(key):
            val = ee.Number(props.get(key)).round()
            return ee.List([key, val])

        updates_tenths = round_tenths.map(round_tenths_fn)
        updates_int = round_int.map(round_int_fn)

        updates_tenths_dict = ee.Dictionary(ee.List(updates_tenths).flatten())
        updates_int_dict = ee.Dictionary(ee.List(updates_int).flatten())

        all_updates = updates_tenths_dict.combine(updates_int_dict, overwrite=True)
        return feat.set(all_updates)

    return feature_collection.map(round_feature)

fc = round_fc(fc)
fc_coarse = round_fc(fc_coarse)


In [38]:
# Test the feature collections

# first_feature = fc.first()
# property_names = first_feature.propertyNames()
# print('Property names:', property_names.getInfo())
# print(first_feature.getInfo())
# print(len(property_names.getInfo()))

# first_coarse_feature = fc_coarse.first()
# coarse_property_names = first_coarse_feature.propertyNames()
# print('Coarse property names:', coarse_property_names.getInfo())
# print(first_coarse_feature.getInfo())
# print(len(coarse_property_names.getInfo()))

In [ ]:
# # Export in batches
# # This needs to be done fully server-side to prevent creation
# # of the entire feature collection in memory, which hits memory limitations


aoi_bounds = aoi.geometry().bounds()
grid = aoi_bounds.coveringGrid(projection=ee.Projection('EPSG:4326').atScale(scale))

# Optional: assign an ID to each grid tile
grid = grid.map(lambda f: f.set('tile_id', f.id()))

# 2. Loop through tiles and filter sample points
grid_list = grid.toList(grid.size())
num_tiles = grid.size().getInfo()

for i in range(num_tiles):
    tile = ee.Feature(grid_list.get(i))
    tile_geom = tile.geometry()
    tile_id = tile.get('tile_id').getInfo()

    fc_tile = fc.filterBounds(tile_geom)

    # 3. Export tile points
    task = ee.batch.Export.table.toDrive(
        collection=fc_tile,
        description=f'fc_tile_{AOI_READABLE}_{tile_id}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'fc_tile_{AOI_READABLE}_{tile_id}',
        fileFormat='CSV'
    )
    task.start()









##################### DEPRECATED - DOESN'T BATCH GEOGRAPHICALLY #####################

# # Get estimated size of the feature collection based on area of forest in AOI

# # Mask the image to retain only forest (value == 1)
# binary = combined_forest_mask.updateMask(combined_forest_mask.eq(1)).rename('forest')

# # Multiply by pixel area to get area per pixel
# area_image = binary.multiply(ee.Image.pixelArea())

# # Reduce to total area in square meters
# area_stats = area_image.reduceRegion(
#     reducer=ee.Reducer.sum(),
#     geometry=aoi,
#     scale=30,
#     maxPixels=1e13
# )

# # Convert to square kilometers
# forest_area_km2 = ee.Number(area_stats.get('forest')).divide(1e6)
# estimated_partitions = forest_area_km2.multiply(FOREST_RANDOM_SAMPLE).divide(BATCH_SIZE).ceil()

# # Add reproducible random column
# fc = fc.randomColumn('batch_rand', seed=RANDOM_SEED)

# # Define number of partitions (based on estimated size or safe guess)
# num_partitions = estimated_partitions.getInfo()
# step = 1.0 / num_partitions
# print(f"Estimated partitions: {num_partitions}, Step size: {step}")

# # Export loop
# for i in range(num_partitions):
#     print(f"Starting export for batch {i + 1} of {num_partitions}")
#     lower = i * step
#     upper = (i + 1) * step
#     subset = fc.filter(ee.Filter.And(
#         ee.Filter.gte('batch_rand', lower),
#         ee.Filter.lt('batch_rand', upper)
#     ))

#     task = ee.batch.Export.table.toDrive(
#         collection=subset,
#         description=f'fc_export_batch_{AOI_READABLE}_btchsz{BATCH_SIZE}_{i}',
#         folder=DRIVE_FOLDER,
#         fileNamePrefix=f'fc_batch_{AOI_READABLE}_btchsz{BATCH_SIZE}_{i}',
#         fileFormat='CSV'
#     )
#     task.start()






########## DEPRECATED - NEED TO BATCH ############
# # Export fine-resolution feature collection
# task_fc = ee.batch.Export.table.toDrive(
#     collection=fc,
#     description=f'fc_{AOI_READABLE}',
#     fileFormat='CSV',
#     folder=DRIVE_FOLDER, 
#     fileNamePrefix=f'fc_{AOI_READABLE}'
# )
# task_fc.start()

# # Export coarse-resolution feature collection
# task_fc_coarse = ee.batch.Export.table.toDrive(
#     collection=fc_coarse,
#     description= f'fc_coarse_{AOI_READABLE}',
#     fileFormat='CSV',
#     folder=DRIVE_FOLDER,
#     fileNamePrefix= f'fc_coarse_{AOI_READABLE}'
# )
# task_fc_coarse.start()


Estimated partitions: 18, Step size: 0.05555555555555555
Starting export for batch 1 of 18
Starting export for batch 2 of 18
Starting export for batch 3 of 18
Starting export for batch 4 of 18
Starting export for batch 5 of 18
Starting export for batch 6 of 18
Starting export for batch 7 of 18
Starting export for batch 8 of 18
Starting export for batch 9 of 18
Starting export for batch 10 of 18
Starting export for batch 11 of 18
Starting export for batch 12 of 18
Starting export for batch 13 of 18
Starting export for batch 14 of 18
Starting export for batch 15 of 18
Starting export for batch 16 of 18
Starting export for batch 17 of 18
Starting export for batch 18 of 18
